# The Negotiation Simulator

### Investment Banking — Banking-AI


            ## Step 0 — Notebook Header

            **Prerequisites:** Basic Python, beginner familiarity with text classification, and comfort reading contract or negotiation language.

            **What You Will Learn:**

            - Describe how historical negotiation language can reveal likely counterparty objection themes.
- Compare a simple keyword baseline against a text model for argument prediction.
- Construct a simulator that recommends term bundles based on predicted negotiation patterns.

            **Estimated time:** 45 minutes

            **Collection context:** Investment-banking notebooks that preserve internal pattern knowledge and turn it into reusable decision support for high-stakes interactions.


## Step 1 — Investment-Banking Problem Introduction

**Why this problem matters:** Teams often repeat the same negotiation battles, so historical language can be used to prepare stronger redirections before the live meeting begins.

**Operational question:** What counterargument is most likely next, and which package of terms should we prepare in response?

**Primary stakeholders:** legal teams, deal teams, restructuring advisors, and negotiation leads

**Decision supported:** Predict likely objection themes and suggest a practical bundle of terms for the next round.

**Comprehension check:** Before looking at the data, write one sentence explaining why predicting the next argument can be more useful than classifying the current one.

**Domain framing note:** This notebook is educational and synthetic. It demonstrates decision support for investment-banking and adjacent institutional workflows, not production approval or investment advice.


## Step 2 — Environment Setup

Use synthetic negotiation statements so the notebook remains shareable while still modeling recurring argument patterns.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (11, 4)
RANDOM_SEED = 42
RNG = np.random.default_rng(RANDOM_SEED)

print("Environment ready for a synthetic negotiation workflow.")


## Step 3 — Data Creation & Context

We simulate historical negotiation statements with recurring themes such as pricing, liability, timing, and governance so the model can learn objection patterns.


In [ ]:
objection_templates = {
    "Pricing": ["the valuation needs to move lower", "the fee stack is too high", "we need economic relief on price"],
    "Liability": ["the indemnity cap is too narrow", "we need stronger protections on liability", "the downside allocation is not acceptable"],
    "Timing": ["the timeline is unrealistic", "we need more diligence time", "closing cannot happen on that schedule"],
    "Governance": ["board approval rights must expand", "we need tighter consent rights", "the governance package is still too loose"],
}
rows = []

for _ in range(1200):
    label = RNG.choice(list(objection_templates))
    tone = RNG.choice(["firm", "measured", "escalated"])
    text = f"{tone} counterparty statement: {RNG.choice(objection_templates[label])}"
    rows.append((text, label))

negotiation_df = pd.DataFrame(rows, columns=["statement_text", "objection_theme"])
print(negotiation_df.head(3).to_string(index=False))


## Step 4 — Exploratory Data Analysis

Inspect the class balance before fitting the model so you know whether the simulator is learning from a broad portfolio or a narrow one.


In [ ]:
sns.countplot(data=negotiation_df, x="objection_theme", color="#1B9E77")
plt.title("Negotiation Objection Themes")
plt.xlabel("Objection Theme")
plt.ylabel("Count")
plt.tight_layout()
plt.show()
plt.close()


Alt-Text: The class distribution chart shows the mix of negotiation themes the simulator learns from, framing the later argument-prediction task.


## Step 5 — Feature Engineering

We keep the model input simple by using the raw statement text directly, which makes it easier to inspect how phrasing affects prediction.


In [ ]:
analysis_df = negotiation_df.copy()
print(analysis_df.head(3).to_string(index=False))


## Step 6 — Baseline Establishment

A basic baseline is a phrase matcher that looks for obvious terms like `price`, `liability`, or `timeline`.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    analysis_df["statement_text"],
    analysis_df["objection_theme"],
    test_size=0.25,
    random_state=RANDOM_SEED,
    stratify=analysis_df["objection_theme"],
)

def keyword_theme(text: str) -> str:
    lowered = text.lower()
    if "price" in lowered or "valuation" in lowered or "economic" in lowered:
        return "Pricing"
    if "liability" in lowered or "indemnity" in lowered or "protections" in lowered:
        return "Liability"
    if "timeline" in lowered or "schedule" in lowered or "time" in lowered:
        return "Timing"
    return "Governance"

baseline_pred = X_test.apply(keyword_theme)
print(f"Baseline accuracy: {accuracy_score(y_test, baseline_pred):.3f}")


## Step 7 — Model Training & Evaluation

Train a text model that can recognize recurring objection patterns even when exact keywords change.


In [ ]:
negotiation_model = Pipeline(
    [
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2), min_df=2)),
        ("classifier", LogisticRegression(max_iter=1200)),
    ]
)
negotiation_model.fit(X_train, y_train)
model_pred = negotiation_model.predict(X_test)

print(f"Model accuracy: {accuracy_score(y_test, model_pred):.3f}")
print(classification_report(y_test, model_pred))


## Step 8 — Interpretability & Explainability

Top-weighted terms show whether the model is learning meaningful negotiation language rather than overfitting to obvious single words.


In [ ]:
vectorizer = negotiation_model.named_steps["tfidf"]
classifier = negotiation_model.named_steps["classifier"]
feature_names = vectorizer.get_feature_names_out()

for class_index, class_name in enumerate(classifier.classes_):
    top_terms = feature_names[np.argsort(classifier.coef_[class_index])[-8:]][::-1]
    print(f"Top signals for {class_name}: {', '.join(top_terms)}")


## Step 9 — Operational Application

Operationally, the simulator should predict the likely argument and recommend a prepared term bundle before the next negotiation round.


In [ ]:
term_bundles = {
    "Pricing": "offer phased pricing with earn-out protection",
    "Liability": "tighten indemnity language but keep cap economics stable",
    "Timing": "trade milestone-based diligence for a shorter signing path",
    "Governance": "offer limited consent rights with clear carve-outs",
}

live_statements = pd.DataFrame(
    {
        "statement_text": [
            "counterparty says the valuation still feels rich and wants more economic relief",
            "counterparty says the board rights and approvals remain too loose",
            "counterparty says the closing schedule is too aggressive for diligence",
        ]
    }
)
live_statements["predicted_theme"] = negotiation_model.predict(live_statements["statement_text"])
live_statements["recommended_bundle"] = live_statements["predicted_theme"].map(term_bundles)
print(live_statements.to_string(index=False))


            ## Step 10 — Summary & Reflection

            **What you accomplished:** You built a synthetic negotiation simulator that predicts likely objection themes and suggests prepared response packages for the next round.

            **Limitations to note:**

            - The text is synthetic and does not capture the political, relational, or drafting nuance of real negotiations.
- The recommended term bundles are illustrative and should not be treated as legal advice.
- Voice mode is represented here as text transcripts rather than true speech interaction.

            **Ethical reflection:** Preparation tools can improve strategy, but they should not be used to manipulate or misrepresent facts in legal or commercial negotiations.

            **Reflection questions:**

            1. Which negotiation theme would require the most human judgment even if the model prediction were accurate?
2. How would you evaluate whether a recommended bundle improved outcomes rather than just sounding persuasive?
3. What internal governance should exist before training a simulator on sensitive contract history?


            ## References

            - Scikit-learn user guide: text classification with linear models.
- General negotiation analysis literature on pattern recognition and preparation.
- Scenario inspiration: contract-portfolio analysis and negotiation strategy workflows in legal and deal teams.
